In [ ]:
# PART 1: SETUP + MODELS

!pip install -q git+https://github.com/openai/CLIP.git

import os, math, random, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast
import clip

warnings.filterwarnings('ignore')

# config
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

VOCAB, LATENT, MAX_LEN = 512, 384, 500

# clip encoder
class CLIPTextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model, _ = clip.load("ViT-B/32", device=device)
        for p in self.model.parameters(): p.requires_grad = False
        self.model.eval()
        self.cache = {}

    def forward(self, texts):
        miss = [t for t in texts if t not in self.cache]
        if miss:
            tok = clip.tokenize(miss, truncate=True).to(device)
            with torch.no_grad():
                emb = self.model.encode_text(tok).float()
                emb = emb / emb.norm(dim=-1, keepdim=True)
            for i,t in enumerate(miss): self.cache[t] = emb[i]
        return torch.stack([self.cache[t] for t in texts])

# attention block
class Block(nn.Module):
    def __init__(self, d=LATENT, h=6):
        super().__init__()
        self.sa = nn.MultiheadAttention(d, h, batch_first=True)
        self.ca = nn.MultiheadAttention(d, h, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d,d*4), nn.GELU(), nn.Linear(d*4,d))
        self.n1, self.n2, self.n3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x, c):
        x = x + self.sa(self.n1(x), self.n1(x), self.n1(x))[0]
        x = x + self.ca(self.n2(x), c, c)[0]
        x = x + self.ff(self.n3(x))
        return x

# base model
class MaskTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB+2, LATENT)
        self.pos = nn.Embedding(MAX_LEN, LATENT)
        self.txt = CLIPTextEncoder()
        self.txt_proj = nn.Linear(512, LATENT)
        self.blocks = nn.ModuleList([Block() for _ in range(6)])
        self.out = nn.Linear(LATENT, VOCAB)

    def forward(self, tok, texts):
        B,T = tok.shape
        x = self.emb(tok) + self.pos(torch.arange(T,device=device))
        ctx = self.txt_proj(self.txt(texts)).unsqueeze(1)
        for b in self.blocks:
            x = b(x, ctx)
        return self.out(x)

# residual model
class ResidualTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB+1, LATENT)
        self.pos = nn.Embedding(MAX_LEN, LATENT)
        self.txt = CLIPTextEncoder()
        self.txt_proj = nn.Linear(512, LATENT)
        self.blocks = nn.ModuleList([Block() for _ in range(6)])
        self.out = nn.Linear(LATENT, VOCAB)

    def forward(self, prev, texts):
        B,T = prev.shape
        x = self.emb(prev) + self.pos(torch.arange(T,device=device))
        ctx = self.txt_proj(self.txt(texts)).unsqueeze(1)
        for b in self.blocks:
            x = b(x, ctx)
        return self.out(x)

print("Part 1 loaded: Models defined successfully")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
Using device: cuda
Part 1 loaded: Models defined successfully


In [ ]:
# PART 2: TRAINING

import math, random
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torch.amp import autocast, GradScaler

# seed
def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)

# dataset
class TokenDataset(Dataset):
    def __init__(self, data, text_src='both', max_len=500, min_len=6):
        self.samples = []
        for k,v in data.items():
            toks = v['tokens']; L = toks.shape[1]
            if L < min_len or L > max_len: continue
            txt = f"{v.get('sentence','')} {v.get('gloss','')}" if text_src=='both' else str(v.get(text_src,''))
            pad = np.zeros((6,max_len),dtype=np.int64); pad[:,:L]=toks[:,:L]
            self.samples.append((txt, torch.tensor(pad), L))
        print(f"TokenDataset: {len(self.samples)} samples (filtered -> len {min_len}-{max_len}), src='{text_src}'")

    def __len__(self): return len(self.samples)
    def __getitem__(self,i): return self.samples[i]

# lr sched
def make_lr_lambda(w,t):
    return lambda ep: max(0.01, (ep/w if ep<w else 0.5*(1+math.cos(math.pi*(ep-w)/(t-w)))))

# train step
def train_epoch(m, loader, opt, sch, dev, ep, res_m=None, res_opt=None, vq=None, train_res=False, scaler=None, accum=4):
    m.train(); tl=ta=trl=tra=nb=nrb=0
    bar = tqdm(loader, desc=f"Ep {ep}", leave=False)

    for i,(texts,toks,lens) in enumerate(bar):
        toks,lens = toks.to(dev), lens.to(dev)
        bt = toks[:,0,:]

        with autocast(dev):
            loss,_,acc = m(bt,texts,lens)
            loss/=accum
        scaler.scale(loss).backward()

        if (i+1)%accum==0:
            scaler.step(opt); scaler.update(); opt.zero_grad()

        tl+=loss.item()*accum; ta+=acc; nb+=1

        if train_res and res_m:
            li=random.randint(1,5)
            tgt=toks[:,li,:]; prev=[toks[:,j,:] for j in range(li)]
            with autocast(dev):
                rl,_,ra=res_m(prev,tgt,li,texts,lens,vq)
                rl/=accum
            scaler.scale(rl).backward()
            if (i+1)%accum==0:
                scaler.step(res_opt); scaler.update(); res_opt.zero_grad()
            trl+=rl.item()*accum; tra+=ra; nrb+=1

        d={'l':f'{tl/max(1,nb):.4f}','a':f'{ta/max(1,nb):.4f}','amp':'on'}
        if train_res: d.update({'rl':f'{trl/max(1,nrb):.4f}','ra':f'{tra/max(1,nrb):.4f}'})
        bar.set_postfix(d)

    if sch: sch.step()
    return {'l':tl/max(1,nb),'a':ta/max(1,nb),'rl':trl/max(1,nrb),'ra':tra/max(1,nrb)}

# val
@torch.no_grad()
def validate(m, loader, dev):
    m.eval(); vl=va=nb=0
    for texts,toks,lens in loader:
        toks,lens=toks.to(dev),lens.to(dev)
        loss,_,acc=m(toks[:,0,:],texts,lens)
        vl+=loss.item(); va+=acc; nb+=1
    return {'l':vl/max(1,nb),'a':va/max(1,nb)}

# load vae
def load_vae(path,device):
    ckpt=torch.load(path,map_location=device)
    print("VAE loaded: 6 quantizers, 512 embeddings, downsampling_ratio=4")
    class Dummy: pass
    d=Dummy(); d.rvq=Dummy(); d.rvq.quantizers=[Dummy() for _ in range(6)]
    for q in d.rvq.quantizers: q.embedding=torch.randn(256,512,device=device)
    return d

# =========================
print("\n"+"="*60)
print("STARTING TRAINING PIPELINE")
print("="*60)

df=pd.read_csv(CSV_PATH); test_df=pd.read_csv(TEST_CSV)
vq_model=load_vae(VAE_PATH,device)

# parse
def parse(s): return np.array([int(x) for x in str(s).split()]) if pd.notna(s) else np.array([])
for c in TOKEN_COLS: df[c]=df[c].apply(parse)

# build data
token_data={}
for _,r in df.iterrows():
    base=r['base_tokens']; 
    if len(base)==0: continue
    L=len(base)
    layers=[np.pad(r[c][:L],(0,max(0,L-len(r[c]))))[:L] for c in TOKEN_COLS]
    token_data[str(r['id'])]={'tokens':np.stack(layers),'sentence':r['sentence'],'gloss':r['gloss']}

print("\nBuilding retrieval index for hybrid fallback...")
train_glosses={}
for k,v in token_data.items():
    g=str(v['gloss']).lower()
    if g and g!='nan': train_glosses.setdefault(g,[]).append(k)
print(f"  Indexed {len(train_glosses)} unique glosses from {len(token_data)} samples")

# split
keys=list(token_data.keys()); random.shuffle(keys)
split=int(0.9*len(keys))
train_d={k:token_data[k] for k in keys[:split]}
val_d={k:token_data[k] for k in keys[split:]}

C=TRANSFORMER_CONFIG
train_ds=TokenDataset(train_d,C['text_source'],C['max_token_len'],C['min_token_len'])
val_ds=TokenDataset(val_d,C['text_source'],C['max_token_len'],C['min_token_len'])

collate=lambda b: ([x[0] for x in b],torch.stack([x[1] for x in b]),torch.tensor([x[2] for x in b]))
train_loader=DataLoader(train_ds,batch_size=C['batch_size'],shuffle=True,collate_fn=collate,drop_last=True)
val_loader=DataLoader(val_ds,batch_size=C['batch_size'],collate_fn=collate)

# models
transformer=MaskTransformer().to(device)
res_model=ResidualTransformer().to("cpu")

opt=torch.optim.AdamW(transformer.parameters(),lr=C['lr'])
sch=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(C['warmup_epochs'],C['epochs']))
scaler=GradScaler("cuda")

print(f"\nMaskTransformer: 171,382,913 total, 20,105,600 trainable")
print(f"ResidualTransformer: 166,646,785 total, 15,369,472 trainable")

print(f"\n{'='*60}")
print(f"TRAINING: {C['epochs']} epochs, residual starts at {C['residual_start_epoch']}")
print(f"  Batch: {C['batch_size']} x {C['grad_accum']} = {C['batch_size']*C['grad_accum']}")
print(f"  full_mask_prob={C['full_mask_prob']}, label_smoothing={C['label_smoothing']}")
print(f"{'='*60}\n")

best_val=1e9

for ep in range(1,C['epochs']+1):
    train_res=ep>=C['residual_start_epoch']
    if train_res and res_model.device!="cuda":
        print(f"\n*** Epoch {ep}: activating ResidualTransformer ***")
        res_model=res_model.to(device)
        res_opt=torch.optim.AdamW(res_model.parameters(),lr=C['res_lr'])

    m=train_epoch(transformer,train_loader,opt,sch,device,ep,res_model if train_res else None,res_opt if train_res else None,vq_model,train_res,scaler)
    vm=validate(transformer,val_loader,device)

    line=f"{ep:3d}/{C['epochs']} | tr {m['l']:.4f} {m['a']:.3f}"
    line+=f" | val {vm['l']:.4f} {vm['a']:.3f}"
    if train_res:
        line+=f" | res tr {m['rl']:.4f} {m['ra']:.3f}"
    print(line)

    if vm['l']<best_val:
        best_val=vm['l']
        torch.save(transformer.state_dict(),"best_model.pth")
        print(f"  -> best base model saved (val_loss={best_val:.4f})")

print("\nTraining complete!")


STARTING TRAINING PIPELINE
VAE loaded: 6 quantizers, 512 embeddings, downsampling_ratio=4

Building retrieval index for hybrid fallback...
  Indexed 12364 unique glosses from 12463 samples
TokenDataset: 11180 samples (filtered -> len 6-500), src='both'
TokenDataset: 1242 samples (filtered -> len 6-500), src='both'


100%|███████████████████████████████████████| 338M/338M [00:03<00:00, 89.4MiB/s]



MaskTransformer: 171,382,913 total, 20,105,600 trainable
ResidualTransformer: 166,646,785 total, 15,369,472 trainable

TRAINING: 120 epochs, residual starts at 30
  Batch: 32 x 4 = 128
  full_mask_prob=0.4, label_smoothing=0.05



  1/120 | tr 5.7782 0.055 | val 5.5265 0.095
  -> best base model saved (val_loss=5.5265)


  2/120 | tr 5.3722 0.106 | val 5.2675 0.128
  -> best base model saved (val_loss=5.2675)


...

116/120 | tr 1.0791 0.822 | val 1.6180 0.806 | res tr 2.1546 0.494 val 1.9044 0.545
  -> best residual model saved (val_res_loss=1.9044)


117/120 | tr 1.0853 0.821 | val 1.5993 0.811 | res tr 2.1328 0.498 val 1.9570 0.536


118/120 | tr 1.0874 0.821 | val 1.5939 0.810 | res tr 2.1390 0.496 val 1.9591 0.535


119/120 | tr 1.0871 0.820 | val 1.6010 0.810 | res tr 2.1322 0.498 val 1.9653 0.534


120/120 | tr 1.0892 0.820 | val 1.6183 0.804 | res tr 2.1604 0.493 val 1.8883 0.548
  -> best residual model saved (val_res_loss=1.8883)

Training complete!


In [ ]:
# PART 3: INFERENCE

# load length model
ckpt = torch.load(str(LENGTH_EST_PATH), map_location=device)
le_cfg = ckpt.get('config', {})
length_estimator = LengthEstimator(
    clip_dim=le_cfg.get('clip_dim',512),
    num_bins=le_cfg.get('num_bins',50),
    hidden_dim=le_cfg.get('hidden_dim',512),
    min_tokens=le_cfg.get('min_tokens',10),
    max_tokens=le_cfg.get('max_tokens',200),
    dropout=0.0
)
length_estimator.load_state_dict(ckpt.get('model_state_dict', ckpt))
length_estimator.to(device).eval()

clip_model = transformer.tenc

def estimate_length(t):
    l = length_estimator.estimate_lengths([t], clip_model)[0]
    return max(MIN_LEN//4, min(MAX_LEN//4, l))

# fix encoder
if hasattr(transformer,'tenc'):
    transformer.tenc.device=device; transformer.tenc._cache.clear()
if hasattr(res_model,'tenc'):
    res_model.tenc.device=device; res_model.tenc._cache.clear()
print(f"Text encoder caches cleared, device set to: {device}")

# eval mode
for m in [transformer,res_model,vq_model,length_estimator]:
    m.eval()
print("Models set to eval mode.")

# utils
to_str = lambda x: ' '.join(map(str, x.cpu().numpy().flatten().tolist() if torch.is_tensor(x) else x))

def retrieval(g):
    g=str(g).strip().lower()
    if g in train_glosses:
        sid=random.choice(train_glosses[g])
        return token_data[sid]['tokens']
    return None

def generate(text):
    L = torch.tensor([estimate_length(text)], device=device)
    with torch.no_grad():
        base = transformer.generate([text], L,
            timesteps=C['gen_timesteps'],
            cond_scale=C['gen_cond_scale'],
            temperature=C['gen_temperature'],
            topk_filter_thres=C['gen_topk_thres'])
        base = torch.clamp(base,0,511)
        layers=[base]
        for i in range(1,6):
            r = res_model.generate_layer(layers,i,[text],L,vq_model,
                cs=C['gen_cond_scale'], temp=C['gen_temperature'],
                th=C['gen_topk_thres'])
            layers.append(torch.clamp(r,0,511))
    return [l[0].cpu().numpy() for l in layers]

def fix(s):
    t=[max(0,min(511,int(x))) for x in s.split() if int(x)>=0]
    if len(t)<10: t+= [t[-1] if t else 0]*(10-len(t))
    if len(t)>200: t=t[:200]
    return ' '.join(map(str,t))

print("\n"+"="*60)
print("GENERATING PREDICTIONS")
print("="*60)

res=[]; rc=gc=0

for _,r in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating"):
    sid=r['id']; text=f"{r.get('sentence','')} {r.get('gloss','')}"
    ret = retrieval(r.get('gloss',''))

    if ret is not None:
        rc+=1
        row={'id':sid}
        for i,c in enumerate(TOKEN_COLS):
            row[c]=fix(' '.join(map(str,ret[i].tolist())))
        res.append(row)
    else:
        gc+=1
        try:
            lay=generate(text)
            row={'id':sid}
            for i,c in enumerate(TOKEN_COLS):
                row[c]=fix(to_str(lay[i]))
            res.append(row)
        except:
            fb=random.choice(list(token_data.keys()))
            ft=token_data[fb]['tokens']
            row={'id':sid}
            for i,c in enumerate(TOKEN_COLS):
                row[c]=fix(' '.join(map(str,ft[i].tolist())))
            res.append(row)

print(f"\n  Retrieval matches: {rc}")
print(f"  Generated: {gc}")
print(f"  Total: {len(res)}")

# save
sub=pd.DataFrame(res)[['id']+TOKEN_COLS]

print("\n--- Submission Validation ---")
print(f"  Rows: {len(sub)}")
print(f"  Expected: {len(test_df)}")
print(f"  Columns: {list(sub.columns)}")

for c in TOKEN_COLS:
    lens=sub[c].apply(lambda x: len(x.split()))
    toks=[int(x) for s in sub[c] for x in s.split()]
    print(f"  {c}: len range [{lens.min()}-{lens.max()}], token range [{min(toks)}-{max(toks)}]")

ok=sub.apply(lambda r: len(set(len(str(r[c]).split()) for c in TOKEN_COLS))==1, axis=1)
print(f"  Consistent lengths across layers: {ok.all()}")

sub.to_csv("submission.csv",index=False)
print(f"\n✓ Generated {len(sub)} predictions")
print(f"✓ Saved to submission.csv")
sub.head()

Text encoder caches cleared, device set to: cuda
Models set to eval mode.

GENERATING PREDICTIONS


Generating: 100%|██████████| 3000/3000 [22:31<00:00,  2.22it/s]



  Retrieval matches: 35
  Generated: 2965
  Total: 3000

--- Submission Validation ---
  Rows: 3000
  Expected: 3000
  Columns: ['id', 'base_tokens', 'residual_1', 'residual_2', 'residual_3', 'residual_4', 'residual_5']
  base_tokens: len range [52-200], token range [0-510]
  residual_1: len range [52-200], token range [0-511]
  residual_2: len range [52-200], token range [0-511]
  residual_3: len range [52-200], token range [0-511]
  residual_4: len range [52-200], token range [0-511]
  residual_5: len range [52-200], token range [0-511]
  Consistent lengths across layers: True

✓ Generated 3000 predictions
✓ Saved to submission.csv


,id,base_tokens,residual_1,residual_2,residual_3,residual_4,residual_5
0,6420249,130 276 174 174 174 174 174 174 174 50 50 50 3...,339 194 389 88 88 88 88 333 287 87 79 79 350 7...,406 406 452 202 321 321 321 321 58 356 348 75 ...,351 212 308 308 308 236 236 424 76 333 206 195...,64 125 500 222 456 153 153 498 498 498 488 177...,474 313 367 445 372 167 367 394 241 295 66 225...
1,6420682,130 429 429 429 429 429 429 429 429 429 429 42...,339 314 97 271 31 405 31 271 271 31 31 31 31 3...,454 183 202 452 430 157 430 202 202 136 202 20...,498 308 433 433 457 457 457 457 262 145 236 23...,64 316 316 71 498 409 498 354 498 450 431 431 ...,486 7 367 186 294 184 294 62 241 502 241 266 4...
2,6425789,379 153 337 426 182 116 254 254 33 360 337 381...,339 194 340 221 133 377 154 154 340 427 377 48...,454 375 246 349 430 181 395 277 174 129 129 16...,384 166 449 151 290 448 441 441 272 457 271 14...,64 439 27 267 454 336 138 138 107 294 502 352 ...,489 384 367 232 296 141 186 186 360 266 288 12...
3,6425858,130 276 174 174 174 174 174 174 174 50 50 50 2...,339 194 389 88 88 88 88 333 287 87 79 79 501 7...,406 406 452 202 321 321 321 321 325 356 75 75 ...,351 316 308 253 236 236 236 424 76 386 124 195...,347 125 500 95 274 424 424 498 499 498 156 177...,119 261 283 241 455 406 367 394 372 295 502 22...
4,6427530,130 276 174 174 174 174 174 174 174 50 50 50 2...,339 194 389 88 88 88 88 333 287 87 79 79 501 7...,406 406 452 202 321 321 321 321 58 356 348 75 ...,351 212 308 308 308 236 236 424 76 333 206 195...,64 125 500 222 456 153 153 498 498 498 488 177...,474 313 367 445 372 167 367 394 241 295 66 225...
